# RAY-IMAGE N4 — Standalone Diagnostic Session

**One session = one experiment.** This notebook runs N4 only. It does **not** retrain N2, N3, or the VAE.

It restores the existing N2 checkpoint from Google Drive, runs N4 diagnostics, and persists results to Drive.

Select a GPU runtime before starting.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. In Colab choose Runtime → Change runtime type → GPU.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
%cd /content
!rm -rf anime-ai-companion
!git clone -q https://github.com/Rishidev-20thcenturey/anime-ai-companion.git
%cd /content/anime-ai-companion
!git fetch -q origin arena/01a07cdc-anime-ai-companion
!git checkout -q arena/01a07cdc-anime-ai-companion
!git rev-parse --short HEAD
!pip install -q -r requirements.txt
print('Active branch ready.')


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/RAY_IMAGE')
DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
DRIVE_N4 = DRIVE_ROOT / 'runs/N4'
DRIVE_N4.mkdir(parents=True, exist_ok=True)

src = DRIVE_CKPT / 'ray_image_v0_2_whiten.pt'
dst = Path('/content/ray_image_v0_2_whiten.pt')
if not src.exists():
    raise FileNotFoundError(f'Persistent N2 checkpoint not found: {src}')
shutil.copy2(src, dst)
print('Restored N2 checkpoint:', dst)
print('N4 Drive output:', DRIVE_N4)


In [ ]:
# N4 ONLY — diagnostic, no training
!rm -rf /content/n4
!python -m ray_image.probe_latent_separability --checkpoint /content/ray_image_v0_2_whiten.pt --outdir /content/n4


In [ ]:
from pathlib import Path
import shutil

report = Path('/content/n4/n4_report.json')
if not report.exists():
    raise FileNotFoundError(report)
print('===== N4 REPORT =====')
print(report.read_text())

target_report = DRIVE_N4 / 'n4_report.json'
shutil.copy2(report, target_report)
grid_src = Path('/content/n4/grids/shape_swap')
grid_dst = DRIVE_N4 / 'grids/shape_swap'
grid_dst.mkdir(parents=True, exist_ok=True)
for p in grid_src.glob('*'):
    if p.is_file(): shutil.copy2(p, grid_dst / p.name)
print('Persistent N4 report:', target_report)
print('Persistent shape-swap grids:', grid_dst)


## Done

Send me the printed N4 report. No training is performed in this notebook.